# Studi Kasus 2 — Fine-Tuning LLM dengan LoRA dan QLoRA

LoRA membekukan bobot dasar W dan mempelajari update ber-rank rendah ΔW=BA. Jika rank r jauh lebih kecil daripada dimensi layer, jumlah parameter trainable turun drastis. QLoRA mempertahankan base model dalam 4-bit dan melatih adapter LoRA pada komputasi presisi lebih tinggi.


In [ ]:
!pip -q install -U transformers peft bitsandbytes accelerate datasets


## Dataset instruction sederhana

Contoh berukuran kecil hanya untuk memeriksa pipeline. Fine-tuning ilmiah memerlukan train/validation/test, keragaman instruksi, deduplikasi, dan evaluasi sebelum–sesudah.


In [ ]:
from datasets import Dataset
examples = [
 {"instruction":"Jelaskan tokenisasi.","response":"Tokenisasi memecah teks menjadi token kata, subword, karakter, atau byte."},
 {"instruction":"Apa fungsi attention?","response":"Attention membobot informasi berdasarkan relevansinya terhadap query."},
 {"instruction":"Apa itu RAG?","response":"RAG mengambil konteks eksternal sebelum model menghasilkan jawaban."},
 {"instruction":"Bedakan precision dan recall.","response":"Precision mengukur ketepatan prediksi positif; recall mengukur cakupan kasus positif."}
]
dataset=Dataset.from_list(examples)


## Memilih LoRA atau QLoRA

Set USE_QLORA=False untuk LoRA pada base model presisi biasa. QLoRA menggunakan NF4 yang dirancang untuk distribusi bobot neural network. Double quantization mengompresi konstanta quantization. Adapter tetap disimpan terpisah dari base model.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID="Qwen/Qwen2.5-0.5B-Instruct"
USE_QLORA=True
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)

quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
) if USE_QLORA else None

model=AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=quant, device_map="auto",
    torch_dtype=None if USE_QLORA else "auto"
)
if USE_QLORA:
    model=prepare_model_for_kbit_training(model)

lora=LoraConfig(
    r=8,lora_alpha=16,lora_dropout=0.05,bias="none",
    task_type="CAUSAL_LM",target_modules=["q_proj","k_proj","v_proj","o_proj"]
)
model=get_peft_model(model,lora)
model.print_trainable_parameters()


## Formatting, tokenisasi, dan label

Prompt chat dibentuk menggunakan template checkpoint. Label sama dengan input IDs, tetapi padding diubah menjadi -100 agar tidak masuk loss.


In [ ]:
def tokenize(example):
    messages=[
      {"role":"system","content":"Anda tutor NLP berbahasa Indonesia."},
      {"role":"user","content":example["instruction"]},
      {"role":"assistant","content":example["response"]}
    ]
    text=tokenizer.apply_chat_template(messages,tokenize=False)
    item=tokenizer(text,truncation=True,max_length=256,padding="max_length")
    item["labels"]=[tok if mask else -100 for tok,mask in zip(item["input_ids"],item["attention_mask"])]
    return item

tokenized=dataset.map(tokenize,remove_columns=dataset.column_names)
tokenized.set_format("torch")


## Training adapter

Epoch banyak pada empat contoh akan overfit. Konfigurasi ini hanya smoke test. Tambahkan validation set dan EarlyStoppingCallback pada eksperimen nyata.


In [ ]:
from transformers import TrainingArguments, Trainer, default_data_collator
args=TrainingArguments(
    output_dir="adapter-nlp",num_train_epochs=1,
    per_device_train_batch_size=1,gradient_accumulation_steps=4,
    learning_rate=2e-4,logging_steps=1,save_strategy="epoch",
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    report_to="none"
)
trainer=Trainer(model=model,args=args,train_dataset=tokenized,data_collator=default_data_collator)
trainer.train()
model.save_pretrained("adapter-nlp-final")


## Evaluasi yang harus ditambahkan

Bandingkan base, LoRA, dan QLoRA pada test prompt yang tidak muncul saat training. Catat validation loss, task metric, trainable parameter ratio, peak GPU memory, waktu training, dan error taxonomy. LoRA vs QLoRA harus memakai dataset, rank, target modules, dan training budget yang sama.
